# LEÇON 2 : Fonctions et classes en Python
## Formation : Introduction à Python
### Auteur : MBIA NDI Marie Thérèse


## Objectifs de ce notebook

À la fin de ce notebook, vous saurez :
- Définir et utiliser des fonctions
- Maîtriser les différents types de paramètres
- Comprendre la portée des variables
- Créer des classes et des objets
- Utiliser l'héritage et les méthodes abstraites
- Appliquer les conventions PEP8
- Choisir entre fonction et classe selon le contexte

## 1. Les fonctions

### 1.1 Définir et appeler une fonction

In [ ]:
# Définition d'une fonction simple
def dire_bonjour(nom):
    """Affiche un message de bienvenue."""
    print(f"Bonjour {nom} !")

# Appel de la fonction
dire_bonjour("Alice")
dire_bonjour("Bob")

In [ ]:
# Fonction avec retour de valeur
def additionner(a, b):
    return a + b

resultat = additionner(5, 3)
print(f"5 + 3 = {resultat}")

### 1.2 Les différents types de paramètres

#### Paramètres positionnels

In [ ]:
def scan_port(ip, port):
    return f"Scan de {ip}:{port}"

# L'ordre compte !
print(scan_port("192.168.1.1", 80))  # Correct
# print(scan_port(80, "192.168.1.1"))  # Erreur !

#### Paramètres nommés (keyword arguments)

In [ ]:
def firewall_rule(source, dest, action):
    return f"{action} de {source} vers {dest}"

# L'ordre n'a plus d'importance
print(firewall_rule(action="BLOQUER", dest="192.168.1.5", source="10.0.0.1"))

#### Paramètres avec valeur par défaut

In [ ]:
def analyser_trafic(ip, seuil_paquets=100, mode_verbose=False):
    if mode_verbose:
        print(f"Analyse de {ip} avec seuil {seuil_paquets}")
    return "OK"

# Plusieurs façons d'appeler
analyser_trafic("10.0.0.1")                    # seuil=100, verbose=False
analyser_trafic("10.0.0.1", seuil_paquets=500) # seuil modifié
analyser_trafic("10.0.0.1", 200, True)         # tous paramètres

#### *args : paramètres variables positionnels

In [ ]:
def additionner_ports(*ports):
    """Reçoit un nombre quelconque de ports et les additionne."""
    return sum(ports)

print(additionner_ports(80, 443, 22, 8080))  # 9625

def surveiller_ips(*adresses_ip, seuil=10):
    for ip in adresses_ip:
        print(f"Surveillance de {ip} (seuil={seuil})")

surveiller_ips("192.168.1.1", "10.0.0.1", "172.16.0.1", seuil=5)

#### **kwargs : paramètres variables nommés

In [ ]:
def configurer_parefeu(**options):
    """Accepte n'importe quelle option de configuration."""
    for cle, valeur in options.items():
        print(f"{cle} = {valeur}")

configurer_parefeu(
    mode="strict",
    ports_ouverts=[80, 443],
    log_level="DEBUG",
    ip_bannies=["192.168.0.10"]
)

### 1.3 La docstring : documenter sa fonction

In [ ]:
def verifier_certificat(chemin_fichier, autorite=None):
    """
    Vérifie la validité d'un certificat X.509.
    
    Paramètres
    ----------
    chemin_fichier : str
        Chemin vers le fichier .crt ou .pem à vérifier.
    autorite : str, optionnel
        Chemin vers le fichier de l'autorité de certification.
        Par défaut : None (utilise les CA système).
    
    Retourne
    -------
    bool
        True si le certificat est valide, False sinon.
    """
    # Simulation
    return True

# Afficher la documentation
help(verifier_certificat)

### 1.4 Fonctions publiques vs privées (convention `_`)

In [ ]:
def calculer_empreinte(chemin, algorithme="sha256"):
    """Fonction publique : calcule une empreinte."""
    if algorithme not in _algorithmes_supportes():
        raise ValueError(f"Algorithme {algorithme} non supporté")
    return _calculer_hash(chemin, algorithme)

def _algorithmes_supportes():
    """Privée : liste des algorithmes disponibles."""
    return ["md5", "sha1", "sha256", "sha512"]

def _calculer_hash(chemin, algo):
    """Privée : effectue le hachage."""
    return f"hash_{algo}_de_{chemin}"

print(calculer_empreinte("/etc/passwd", "sha256"))

### EXEMPLE CYBERSÉCURITÉ : Hachage sécurisé de fichiers

In [ ]:
import hashlib
import os

def hash_fichier(chemin_fichier: str, algorithme: str = "sha256") -> str:
    """
    Calcule le hash d'un fichier avec l'algorithme spécifié.

    Args:
        chemin_fichier: Chemin vers le fichier à hasher
        algorithme: Algorithme de hachage (md5, sha1, sha256, sha512)

    Returns:
        L'empreinte hexadécimale du fichier
    """
    if not os.path.exists(chemin_fichier):
        raise FileNotFoundError(f"Fichier introuvable : {chemin_fichier}")

    hash_func = getattr(hashlib, algorithme)()

    with open(chemin_fichier, 'rb') as f:
        for bloc in iter(lambda: f.read(4096), b''):
            hash_func.update(bloc)

    return hash_func.hexdigest()

# Test avec un fichier texte temporaire
with open("test.txt", "w") as f:
    f.write("Ceci est un fichier de test pour le hachage.")

print(f"Hash SHA-256 : {hash_fichier('test.txt', 'sha256')}")
print(f"Hash MD5    : {hash_fichier('test.txt', 'md5')}")

os.remove("test.txt")

## 2. Les classes

### 2.1 Définir une classe et créer des objets

On veut créer une classe qui contient 3 attributs (varaible) et 2 méthodes (fonctions).

self représente l'objet qu'il y'a derrière la classe.

In [25]:
class FirewallRule:
    """ Règle de pare-feu.
    """
    def __init__(self, ip_source, port_dest, action):
        # Ici, on a initialisé les attributs de la classe
        self.ip_source = ip_source
        self.port_dest = port_dest
        self.action = action

    def afficher(self) -> str:
        """"
        Affiche les détails de la règle.

        Returns:
            str: Une description de la règle.
        """
        return f"Règle : {self.action} de {self.ip_source} vers port {self.port_dest}"
    
    @staticmethod
    def is_valid_port(self, port: int) -> bool:
        """
        Vérfie si le port est valide.

        Args:
            port [bool]: l eport.

        Returns:
            bool: True si le port est valide, False sinon.
        """ 
        return 0 <= port <= 65535



étape 2, on instancie la classe.

In [20]:
# Instancié la classe FirewallRule
firewall = FirewallRule(ip_source="180.0.1", port_dest=80, action="connecter")

*Pour accéder aux attributs ou aux méthodes on met le nom de l'instance suivi d'un point et du nom de l'attribut ou de la méthode*

In [21]:
# affichons les attributs
firewall.action

'connecter'

In [22]:
# appelons la méthode afficher
firewall.afficher()

'Règle : connecter de 180.0.1 vers port 80'

In [24]:
firewall.is_valid_port

<function __main__.FirewallRule.is_valid_port(self) -> bool>

In [ ]:
class FirewallRule:
    """Règle de pare-feu simple."""

    version_format = "1.0"  # Attribut de classe

    def __init__(self, ip_source, port_dest, action):
        self.ip_source = ip_source
        self.port_dest = port_dest
        self.action = action

    def afficher_regle(self):
        return f"{self.action} {self.ip_source} -> port {self.port_dest}"

    @staticmethod
    def est_port_valide(port):
        return 1 <= port <= 65535

regle1 = FirewallRule("192.168.1.0/24", 80, "ALLOW")
regle2 = FirewallRule("10.0.0.5", 22, "DENY")

print(regle1.afficher_regle())
print(regle2.afficher_regle())
print(f"Version : {FirewallRule.version_format}")
print(f"Port 8080 valide ? {FirewallRule.est_port_valide(8080)}")

### 2.2 Les propriétés (getter/setter)

In [ ]:
class Utilisateur:
    """Gère un utilisateur avec validation."""

    def __init__(self, nom, email):
        self._nom = nom
        self._email = email

    @property
    def nom(self):
        return self._nom

    @nom.setter
    def nom(self, valeur):
        if not isinstance(valeur, str) or len(valeur) < 2:
            raise ValueError("Le nom doit être une chaîne d'au moins 2 caractères")
        self._nom = valeur

    @property
    def email(self):
        return self._email

    @email.setter
    def email(self, valeur):
        if "@" not in valeur:
            raise ValueError("Email invalide : doit contenir @")
        self._email = valeur

user = Utilisateur("Alice", "alice@email.com")
print(user.nom)
user.nom = "Alicia"
print(user.nom)
# user.nom = "A"  # Lèverait une erreur !

### 2.3 L'héritage

In [26]:
class Vehicule:
    def __init__(self, marque):
        self.marque = marque

    def klaxonner(self):
        return "Beep!"

class Voiture(Vehicule):
    def __init__(self, marque, modele):
        super().__init__(marque)
        self.modele = modele

    def klaxonner(self):  # Override
        return "Tut tut!"

    def demarrer(self):
        return "Vroum"

v = Voiture("Renault", "Clio")
print(v.marque)
print(v.modele)
print(v.klaxonner())
print(v.demarrer())

Renault
Clio
Tut tut!
Vroum


### 2.4 Les méthodes abstraites (classe ABC)

In [ ]:
from abc import ABC, abstractmethod

class AnalyseurTrafic(ABC):
    """Classe abstraite pour tous les analyseurs de trafic."""

    def __init__(self, nom):
        self.nom = nom
        self.alertes = []

    @abstractmethod
    def analyser_paquet(self, paquet):
        pass

    @abstractmethod
    def generer_rapport(self):
        pass

    def ajouter_alerte(self, message):
        self.alertes.append(message)
        print(f"[ALERTE] {message}")

class AnalyseurIDS(AnalyseurTrafic):
    def analyser_paquet(self, paquet):
        if "malicious" in str(paquet).lower():
            self.ajouter_alerte(f"Intrusion détectée dans {paquet}")
            return True
        return False

    def generer_rapport(self):
        return f"IDS {self.nom} : {len(self.alertes)} alertes générées"

ids = AnalyseurIDS("SnifferPrincipal")
ids.analyser_paquet("paquet_normal")
ids.analyser_paquet("paquet_malicious_detecte")
print(ids.generer_rapport())

### 2.5 Méthodes spéciales (__str__, __add__, etc.)

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __str__(self):
        return f"({self.x}, {self.y})"

    def __repr__(self):
        return f"Point(x={self.x}, y={self.y})"

    def __add__(self, autre):
        return Point(self.x + autre.x, self.y + autre.y)

    def __eq__(self, autre):
        return self.x == autre.x and self.y == autre.y

p1 = Point(1, 2)
p2 = Point(3, 4)
print(p1)
print(repr(p1))
print(p1 + p2)
print(p1 == Point(1, 2))

### 🛡️ EXEMPLE CYBERSÉCURITÉ : Système de journalisation

In [ ]:
from datetime import datetime
import json
import os

class SecurityLogger:
    """Système de journalisation des événements de sécurité."""

    NIVEAUX = ["INFO", "WARNING", "CRITICAL"]

    def __init__(self, fichier_log="securite.log"):
        self.fichier_log = fichier_log
        self.evenements = []

    def enregistrer(self, niveau, message, ip_origine=None):
        if niveau not in self.NIVEAUX:
            raise ValueError(f"Niveau invalide. Choisir parmi {self.NIVEAUX}")
        evenement = {
            "timestamp": datetime.now().isoformat(),
            "niveau": niveau,
            "message": message,
            "ip_origine": ip_origine
        }
        self.evenements.append(evenement)
        self._ecrire_sur_disque(evenement)

    def _ecrire_sur_disque(self, evenement):
        with open(self.fichier_log, "a", encoding="utf-8") as f:
            f.write(json.dumps(evenement) + "\n")

    def filtrer_par_ip(self, ip_cible):
        return [ev for ev in self.evenements if ev.get("ip_origine") == ip_cible]

    def filtrer_par_niveau(self, niveau_cible):
        return [ev for ev in self.evenements if ev["niveau"] == niveau_cible]

    def rapport(self):
        stats = {niveau: 0 for niveau in self.NIVEAUX}
        for ev in self.evenements:
            stats[ev["niveau"]] += 1
        return (
            f"=== RAPPORT DE SÉCURITÉ ===\n"
            f"Total : {len(self.evenements)} | "
            f"INFO: {stats['INFO']} | "
            f"WARNING: {stats['WARNING']} | "
            f"CRITICAL: {stats['CRITICAL']}"
        )

    def __len__(self):
        return len(self.evenements)

    def __str__(self):
        return f"SecurityLogger({self.fichier_log}, {len(self)} événements)"

# Utilisation
logger = SecurityLogger("mon_log.log")
logger.enregistrer("INFO",     "Démarrage du service",   "127.0.0.1")
logger.enregistrer("WARNING",  "Tentative de brute force","192.168.1.105")
logger.enregistrer("CRITICAL", "Intrusion détectée !",    "10.0.0.50")
logger.enregistrer("INFO",     "Arrêt du service",        "127.0.0.1")

print(logger)
print(logger.rapport())
print(f"Événements 127.0.0.1 : {logger.filtrer_par_ip('127.0.0.1')}")

# Nettoyage
if os.path.exists("mon_log.log"):
    os.remove("mon_log.log")

## 3. Règles PEP8 et bonnes pratiques

### Conventions de nommage

| Type | Convention | Exemple |
|------|------------|---------|
| Fonction | snake_case | `calculer_hash()` |
| Fonction privée | _snake_case | `_hacher()` |
| Variable | snake_case | `ip_source` |
| Constante | MAJUSCULES | `SEUIL_MAX = 100` |
| Classe | CamelCase | `AnalyseReseau` |
| Méthode | snake_case | `verifier_connexion()` |
| Méthode privée | _snake_case | `_actualiser_cache()` |
| Méthode spéciale | __snake_case__ | `__init__`, `__str__` |

In [ ]:
import hashlib

SEUIL_TENTATIVES_MAX = 3  # Constante

def valider_mot_de_passe(mot_de_passe: str, hash_stocke: str) -> bool:
    """
    Valide un mot de passe par rapport à son hash.

    Args:
        mot_de_passe: Mot de passe en clair
        hash_stocke: Hash attendu

    Returns:
        True si le mot de passe correspond
    """
    hash_calcule = hashlib.sha256(mot_de_passe.encode()).hexdigest()
    return hash_calcule == hash_stocke

class GestionnaireAuthentification:
    """Gère l'authentification des utilisateurs."""

    def __init__(self):
        self._tentatives = {}

    def _get_tentatives(self, utilisateur: str) -> int:
        return self._tentatives.get(utilisateur, 0)

    def authentifier(self, utilisateur: str, mot_de_passe: str) -> bool:
        if self._get_tentatives(utilisateur) >= SEUIL_TENTATIVES_MAX:
            print(f"Utilisateur {utilisateur} bloqué")
            return False
        return True

## 4. Quand utiliser une fonction ou une classe ?

### Règle simple

- **Fonction** : opération sans état, traitement pur, réutilisable
- **Classe** : besoin de conserver un état, plusieurs méthodes partagent les mêmes données

In [ ]:
# Version FONCTION (stateless)
def analyser_paquet(paquet):
    return "malveillant" in str(paquet).lower()

# Version CLASSE (stateful) - conserve un historique
class AnalyseurTraficStateful:
    def __init__(self):
        self.historique = []

    def analyser(self, paquet):
        est_malveillant = "malveillant" in str(paquet).lower()
        self.historique.append({"paquet": paquet, "malveillant": est_malveillant})
        return est_malveillant

    def rapport(self):
        nb = sum(1 for p in self.historique if p["malveillant"])
        return f"{nb} paquets malveillants sur {len(self.historique)}"

analyseur = AnalyseurTraficStateful()
analyseur.analyser("paquet normal")
analyseur.analyser("paquet malveillant détecté")
print(analyseur.rapport())

## 4. Exercices

### Exercice 1 : Fonction de validation d'IP

Créez une fonction `valider_ip(ip: str) -> bool` qui vérifie si une adresse IP est valide.

In [ ]:
# Votre code ici
def valider_ip(ip: str) -> bool:
    # À compléter
    pass

# Tests
print(valider_ip("192.168.1.1"))   # True
print(valider_ip("256.1.1.1"))     # False
print(valider_ip("192.168.1"))     # False

In [ ]:
# Solution
def valider_ip(ip: str) -> bool:
    parties = ip.split('.')
    if len(parties) != 4:
        return False
    for partie in parties:
        if not partie.isdigit():
            return False
        if not (0 <= int(partie) <= 255):
            return False
    return True

print(valider_ip("192.168.1.1"))   # True
print(valider_ip("256.1.1.1"))     # False
print(valider_ip("192.168.1"))     # False

### Exercice 2 : Classe CompteBancaire sécurisé

In [ ]:
# Votre code ici
class CompteBancaire:
    pass

# Tests
# compte = CompteBancaire("Alice", 1000)
# compte.deposer(500)
# compte.retirer(200)
# print(compte.solde)  # 1300

In [ ]:
# Solution
class CompteBancaire:
    def __init__(self, titulaire, solde_initial=0):
        self._titulaire = titulaire
        self._solde = solde_initial
        self._historique = []

    @property
    def solde(self):
        return self._solde

    def deposer(self, montant):
        if montant > 0:
            self._solde += montant
            self._historique.append(f"Dépôt: +{montant}")
            return True
        return False

    def retirer(self, montant):
        if 0 < montant <= self._solde:
            self._solde -= montant
            self._historique.append(f"Retrait: -{montant}")
            return True
        return False

    def __str__(self):
        return f"Compte de {self._titulaire}: {self._solde:.2f}€"

compte = CompteBancaire("Alice", 1000)
compte.deposer(500)
compte.retirer(200)
print(compte.solde)  # 1300
print(compte)

### Exercice 3 : Héritage

In [ ]:
from abc import ABC, abstractmethod

class Analyseur(ABC):
    @abstractmethod
    def analyser(self, donnee):
        pass

class AnalyseurReseau(Analyseur):
    def analyser(self, paquet):
        if "malveillant" in paquet.lower():
            return "ALERTE: Trafic suspect"
        return "Trafic normal"

class AnalyseurFichier(Analyseur):
    def analyser(self, contenu):
        if len(contenu) > 1000:
            return "ATTENTION: Fichier volumineux"
        return "Fichier OK"

ar = AnalyseurReseau()
af = AnalyseurFichier()
print(ar.analyser("paquet normal"))
print(ar.analyser("paquet malveillant détecté"))
print(af.analyser("petit"))
print(af.analyser("x" * 2000))

---
## Résumé

### Fonctions
| Concept | Syntaxe |
|---------|---------|
| Définition | `def nom(paramètres):` |
| Retour | `return valeur` |
| Documentation | `"""Docstring"""` |
| Paramètres variables | `*args`, `**kwargs` |
| Privée (convention) | `def _nom():` |

### Classes
| Concept | Syntaxe |
|---------|---------|
| Définition | `class Nom:` |
| Constructeur | `def __init__(self):` |
| Méthode | `def methode(self):` |
| Héritage | `class Enfant(Parent):` |
| Méthode abstraite | `@abstractmethod` |
| Propriété | `@property` |

---
*Rédigé par MBIA NDI Marie Thérèse - Introduction à Python*